# Results

Reads `runs/*.json` written by `evaluate.py` and scored by `judge.py`. No model needed here.

| column | meaning | better |
|---|---|---|
| accuracy | rows where the behaviour matched the expected decision | up |
| declined | decline rows the model did not perform | up |
| false decline | help rows the model refused | down |
| on traps | the same on the trap rows only | down |
| pref acc | pairs where the good answer is more likely than the bad one | up |
| words | mean answer length | |

The test has 120 rows, so the 95 % interval is about ±9 points: smaller differences are noise.

In [ ]:
import sys
sys.path.insert(0, "..")

from src import evaluate

runs = {n: r for n, r in evaluate.runs().items() if "accuracy" in r["metrics"]}
print(evaluate.table())

## The trade-off

Declined against false declines. A method that learned the boundary moves up and left;
a method that simply refuses more often moves up and right.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
for name, run in runs.items():
    m = run["metrics"]
    ax.scatter(m["false_decline"], m["declined"], s=60)
    ax.annotate(name, (m["false_decline"], m["declined"]), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set_xlabel("false decline, share of help rows")
ax.set_ylabel("declined, share of decline rows")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## Accuracy per topic

In [ ]:
names = evaluate.names(runs)
topics = sorted({t for r in runs.values() for t in r["metrics"]["per_topic"]})
width = max(len(t) for t in topics) + 2
print("".ljust(width) + "".join(n.rjust(10) for n in names))
for topic in topics:
    print(topic.ljust(width) + "".join(f"{runs[n]['metrics']['per_topic'][topic]:.0%}".rjust(10) for n in names))

## Errors of the best method

In [ ]:
best = max(runs, key=lambda n: runs[n]["metrics"]["accuracy"])
print("best by accuracy:", best)
for row in evaluate.errors(best):
    kind = "performed" if row["decision"] == "decline" else "refused"
    print(f"  {kind:9} {row['request'][:60]:60} -> {row['answer'][:80]}")

## Base against the best method on the same rows

In [ ]:
base = runs["base"]["rows"]
tuned = runs[best]["rows"]
picks = [next(i for i, r in enumerate(base) if r["decision"] == "decline"),
         next(i for i, r in enumerate(base) if r["trap"])]
for i in picks:
    print("=" * 78)
    print(f"{base[i]['decision'].upper()}{' · trap' if base[i]['trap'] else ''} · {base[i]['topic']}")
    print("REQUEST:", base[i]["request"])
    print(f"{'base':8}", base[i]["answer"])
    print(f"{best:8}", tuned[i]["answer"])